Problem Statement:- Travel planning is not difficult because information is unavailable; it is difficult because information is fragmented, constantly changing, and rarely interpreted according to the individual traveler. Our system addresses this problem by combining destination information, weather, nearby places, user preferences, previous preferences and destination-image understanding into a context-aware multimodal AI pipeline for personalized itinerary generation.

1.Traveltourism Install libraries required

In [2]:
!pip -q install -U google-genai gradio



2.Traveltourism Import required Libraries

In [3]:
import os
import json
import requests

import pandas as pd
from PIL import Image
from getpass import getpass

import gradio as gr
from google import genai


3.API Keysetup and Assigning Model name

In [4]:
from google.colab import userdata
from google import genai

# Retrieve API key from Google Colab Secrets
GEMINI_API_KEY = userdata.get("api_key2")

# Create Gemini client
client = genai.Client(api_key=GEMINI_API_KEY)

# Gemini model
MODEL_NAME = "gemini-3.6-flash"

print("✅ Gemini client configured successfully.")


✅ Gemini client configured successfully.


In [5]:
try:
    test_response = client.models.generate_content(
        model=MODEL_NAME,
        contents="Create a one-sentence personalized travel tip for kerla.")

    print("✅ SUCCESS!")
    print(test_response.text)

except Exception as e:
    print("❌ ERROR")
    print("Error type:", type(e).__name__)
    print("Error message:", str(e))


✅ SUCCESS!
To truly experience the magic of Kerala, balance a serene overnight houseboat stay through the Alleppey backwaters with a refreshing morning walk among Munnar's lush, misty tea plantations.


4.Dataset from Destination

In [6]:
data = {
    "destination": [
        "Bengaluru",
        "Goa",
        "Mysuru",
        "Ooty",
        "Manali",
        "Jaipur",
        "Munnar",
        "Coorg"
    ],

    "state": [
        "Karnataka",
        "Goa",
        "Karnataka",
        "Tamil Nadu",
        "Himachal Pradesh",
        "Rajasthan",
        "Kerala",
        "Karnataka"
    ],

    "destination_type": [
        "City",
        "Beach",
        "Heritage",
        "Hill Station",
        "Mountain",
        "Heritage",
        "Hill Station",
        "Nature"
    ],

    "interests": [
        "Food, Technology, Parks, History",
        "Beach, Food, Nightlife, Water Sports",
        "History, Palace, Culture, Food",
        "Nature, Mountains, Tea, Relaxation",
        "Mountains, Adventure, Snow, Nature",
        "History, Culture, Forts, Shopping",
        "Tea, Nature, Mountains, Waterfalls",
        "Coffee, Nature, Trekking, Relaxation"
    ],

    "nearby_places": [
        "Cubbon Park, Bangalore Palace, Lalbagh, Vidhana Soudha",
        "Baga Beach, Fort Aguada, Calangute Beach, Panjim",
        "Mysore Palace, Chamundi Hill, Brindavan Gardens, Devaraja Market",
        "Ooty Lake, Botanical Garden, Doddabetta Peak, Tea Museum",
        "Solang Valley, Rohtang Pass, Hadimba Temple, Old Manali",
        "Amber Fort, City Palace, Hawa Mahal, Jantar Mantar",
        "Munnar Tea Gardens, Eravikulam National Park, Mattupetty Dam, Attukad Waterfalls",
        "Abbey Falls, Raja's Seat, Dubare, Mandalpatti"
    ],

    "recommended_activities": [
        "City sightseeing, food tours, park visits, technology experiences",
        "Beach activities, water sports, nightlife, seafood experiences",
        "Palace visits, heritage tours, cultural experiences, local food",
        "Tea plantation visits, sightseeing, nature walks, relaxation",
        "Trekking, skiing, snow activities, mountain sightseeing",
        "Fort visits, heritage walks, shopping, cultural experiences",
        "Tea plantation tours, waterfall visits, trekking, wildlife",
        "Coffee plantation tours, trekking, waterfalls, nature walks"
    ],

    "budget_level": [
        "Medium",
        "Medium",
        "Low",
        "Medium",
        "High",
        "Medium",
        "Medium",
        "Medium"
    ],

    "best_for": [
        "Foodies, technology enthusiasts, families",
        "Beach lovers, foodies, adventure seekers, nightlife lovers",
        "History lovers, families, culture enthusiasts",
        "Nature lovers, couples, relaxation seekers",
        "Adventure seekers, mountain lovers, snow enthusiasts",
        "History lovers, culture enthusiasts, shoppers",
        "Nature lovers, trekkers, tea enthusiasts",
        "Nature lovers, coffee enthusiasts, trekkers"
    ],

    "weather_suitability": [
        "Pleasant for city exploration",
        "Best for beach and outdoor activities",
        "Suitable for sightseeing and heritage activities",
        "Cool weather, suitable for relaxation",
        "Cold weather, suitable for snow and adventure",
        "Warm and dry, suitable for heritage sightseeing",
        "Cool and pleasant, suitable for nature activities",
        "Cool and pleasant, suitable for nature and trekking"
    ]
}

destination_df = pd.DataFrame(data)

display(destination_df)


,destination,state,destination_type,interests,nearby_places,recommended_activities,budget_level,best_for,weather_suitability
0,Bengaluru,Karnataka,City,"Food, Technology, Parks, History","Cubbon Park, Bangalore Palace, Lalbagh, Vidhan...","City sightseeing, food tours, park visits, tec...",Medium,"Foodies, technology enthusiasts, families",Pleasant for city exploration
1,Goa,Goa,Beach,"Beach, Food, Nightlife, Water Sports","Baga Beach, Fort Aguada, Calangute Beach, Panjim","Beach activities, water sports, nightlife, sea...",Medium,"Beach lovers, foodies, adventure seekers, nigh...",Best for beach and outdoor activities
2,Mysuru,Karnataka,Heritage,"History, Palace, Culture, Food","Mysore Palace, Chamundi Hill, Brindavan Garden...","Palace visits, heritage tours, cultural experi...",Low,"History lovers, families, culture enthusiasts",Suitable for sightseeing and heritage activities
3,Ooty,Tamil Nadu,Hill Station,"Nature, Mountains, Tea, Relaxation","Ooty Lake, Botanical Garden, Doddabetta Peak, ...","Tea plantation visits, sightseeing, nature wal...",Medium,"Nature lovers, couples, relaxation seekers","Cool weather, suitable for relaxation"
4,Manali,Himachal Pradesh,Mountain,"Mountains, Adventure, Snow, Nature","Solang Valley, Rohtang Pass, Hadimba Temple, O...","Trekking, skiing, snow activities, mountain si...",High,"Adventure seekers, mountain lovers, snow enthu...","Cold weather, suitable for snow and adventure"
5,Jaipur,Rajasthan,Heritage,"History, Culture, Forts, Shopping","Amber Fort, City Palace, Hawa Mahal, Jantar Ma...","Fort visits, heritage walks, shopping, cultura...",Medium,"History lovers, culture enthusiasts, shoppers","Warm and dry, suitable for heritage sightseeing"
6,Munnar,Kerala,Hill Station,"Tea, Nature, Mountains, Waterfalls","Munnar Tea Gardens, Eravikulam National Park, ...","Tea plantation tours, waterfall visits, trekki...",Medium,"Nature lovers, trekkers, tea enthusiasts","Cool and pleasant, suitable for nature activities"
7,Coorg,Karnataka,Nature,"Coffee, Nature, Trekking, Relaxation","Abbey Falls, Raja's Seat, Dubare, Mandalpatti","Coffee plantation tours, trekking, waterfalls,...",Medium,"Nature lovers, coffee enthusiasts, trekkers","Cool and pleasant, suitable for nature and tre..."


5.Cleaning of dataset

In [7]:
# Remove duplicate rows
destination_df = destination_df.drop_duplicates()

# Remove rows with missing destination names
destination_df = destination_df.dropna(
    subset=["destination"]
)

# Columns containing text that need cleaning
text_columns = [
    "destination",
    "state",
    "destination_type",
    "interests",
    "nearby_places",
    "recommended_activities",
    "budget_level",
    "best_for",
    "weather_suitability"
]

# Remove unnecessary spaces from all text columns
for column in text_columns:
    destination_df[column] = (
        destination_df[column]
        .astype(str)
        .str.strip()
    )

# Standardize destination and state names
destination_df["destination"] = (
    destination_df["destination"].str.title()
)

destination_df["state"] = (
    destination_df["state"].str.title()
)

# Standardize destination types
destination_df["destination_type"] = (
    destination_df["destination_type"].str.title()
)

# Standardize budget values
destination_df["budget_level"] = (
    destination_df["budget_level"].str.title()
)

# Reset index after cleaning
destination_df = destination_df.reset_index(drop=True)

print("✅ Dataset cleaned successfully.")
print(f"Total destinations: {len(destination_df)}")

display(destination_df)


✅ Dataset cleaned successfully.
Total destinations: 8


,destination,state,destination_type,interests,nearby_places,recommended_activities,budget_level,best_for,weather_suitability
0,Bengaluru,Karnataka,City,"Food, Technology, Parks, History","Cubbon Park, Bangalore Palace, Lalbagh, Vidhan...","City sightseeing, food tours, park visits, tec...",Medium,"Foodies, technology enthusiasts, families",Pleasant for city exploration
1,Goa,Goa,Beach,"Beach, Food, Nightlife, Water Sports","Baga Beach, Fort Aguada, Calangute Beach, Panjim","Beach activities, water sports, nightlife, sea...",Medium,"Beach lovers, foodies, adventure seekers, nigh...",Best for beach and outdoor activities
2,Mysuru,Karnataka,Heritage,"History, Palace, Culture, Food","Mysore Palace, Chamundi Hill, Brindavan Garden...","Palace visits, heritage tours, cultural experi...",Low,"History lovers, families, culture enthusiasts",Suitable for sightseeing and heritage activities
3,Ooty,Tamil Nadu,Hill Station,"Nature, Mountains, Tea, Relaxation","Ooty Lake, Botanical Garden, Doddabetta Peak, ...","Tea plantation visits, sightseeing, nature wal...",Medium,"Nature lovers, couples, relaxation seekers","Cool weather, suitable for relaxation"
4,Manali,Himachal Pradesh,Mountain,"Mountains, Adventure, Snow, Nature","Solang Valley, Rohtang Pass, Hadimba Temple, O...","Trekking, skiing, snow activities, mountain si...",High,"Adventure seekers, mountain lovers, snow enthu...","Cold weather, suitable for snow and adventure"
5,Jaipur,Rajasthan,Heritage,"History, Culture, Forts, Shopping","Amber Fort, City Palace, Hawa Mahal, Jantar Ma...","Fort visits, heritage walks, shopping, cultura...",Medium,"History lovers, culture enthusiasts, shoppers","Warm and dry, suitable for heritage sightseeing"
6,Munnar,Kerala,Hill Station,"Tea, Nature, Mountains, Waterfalls","Munnar Tea Gardens, Eravikulam National Park, ...","Tea plantation tours, waterfall visits, trekki...",Medium,"Nature lovers, trekkers, tea enthusiasts","Cool and pleasant, suitable for nature activities"
7,Coorg,Karnataka,Nature,"Coffee, Nature, Trekking, Relaxation","Abbey Falls, Raja's Seat, Dubare, Mandalpatti","Coffee plantation tours, trekking, waterfalls,...",Medium,"Nature lovers, coffee enthusiasts, trekkers","Cool and pleasant, suitable for nature and tre..."


6.Filter the destination based on fields such as destination_type, nearby_places, recommended_activities, budget_level, best_for, and weather_suitability, the cleaning code should also handle all relevant fields.

In [8]:
def get_destination_info(destination):
    """
    Retrieve destination information from the destination dataset.

    This function provides the static destination context that will later
    be combined with dynamic weather, nearby places, user preferences,
    previous preferences, and destination-image understanding.
    """

    if not destination or not isinstance(destination, str):
        return None

    # Normalize user input
    destination_query = destination.strip().lower()

    # Search destination
    result = destination_df[
        destination_df["destination"]
        .str.strip()
        .str.lower()
        == destination_query
    ]

    # Destination not found
    if result.empty:
        return None

    # Convert the first matching row to a dictionary
    destination_info = result.iloc[0].to_dict()

    return destination_info


In [9]:
destination_info = get_destination_info("Munnar")

if destination_info:
    print("✅ Destination found!")
    print("📍 Kerala Destination Information:\n")
    print(json.dumps(destination_info, indent=4, ensure_ascii=False))
else:
    print("❌ Destination not found.")



✅ Destination found!
📍 Kerala Destination Information:

{
    "destination": "Munnar",
    "state": "Kerala",
    "destination_type": "Hill Station",
    "interests": "Tea, Nature, Mountains, Waterfalls",
    "nearby_places": "Munnar Tea Gardens, Eravikulam National Park, Mattupetty Dam, Attukad Waterfalls",
    "recommended_activities": "Tea plantation tours, waterfall visits, trekking, wildlife",
    "budget_level": "Medium",
    "best_for": "Nature lovers, trekkers, tea enthusiasts",
    "weather_suitability": "Cool and pleasant, suitable for nature activities"
}


7.Weather API(to get weather update)

In [10]:
# ==========================================
# CELL 7: Weather API
# ==========================================

def get_weather(destination):

    geo_url = "https://geocoding-api.open-meteo.com/v1/search"

    geo_params = {
        "name": destination,
        "count": 1,
        "language": "en",
        "format": "json"
    }

    # -----------------------------
    # STEP 1: Get coordinates
    # -----------------------------

    try:
        geo_response = requests.get(
            geo_url,
            params=geo_params,
            timeout=30
        )

        geo_response.raise_for_status()
        geo_data = geo_response.json()

        if not geo_data.get("results"):
            print("⚠️ Location not found for:", destination)
            return None

        location = geo_data["results"][0]

        latitude = location["latitude"]
        longitude = location["longitude"]

    except requests.exceptions.RequestException as e:
        print("⚠️ Weather location service unavailable.")
        print("Reason:", str(e))
        return None

    # -----------------------------
    # STEP 2: Get 5-day weather
    # -----------------------------

    weather_url = "https://api.open-meteo.com/v1/forecast"

    weather_params = {
        "latitude": latitude,
        "longitude": longitude,
        "current": (
            "temperature_2m,"
            "relative_humidity_2m,"
            "weather_code"
        ),
        "daily": (
            "temperature_2m_max,"
            "temperature_2m_min,"
            "precipitation_probability_max"
        ),
        "forecast_days": 5,
        "timezone": "auto"
    }

    try:
        weather_response = requests.get(
            weather_url,
            params=weather_params,
            timeout=30
        )

        weather_response.raise_for_status()

        return weather_response.json()

    except requests.exceptions.RequestException as e:
        print("⚠️ Weather forecast unavailable.")
        print("Reason:", str(e))
        return None




8.weather Data

In [11]:
# ==========================================
# CELL 8: Simplify Weather Data
# ==========================================

def simplify_weather(weather):

    if weather is None:
        return {
            "status": "Live weather information is currently unavailable."
        }

    current = weather.get("current", {})
    daily = weather.get("daily", {})

    return {
        "status": "Weather data retrieved successfully.",

        "current_temperature": current.get(
            "temperature_2m"
        ),

        "humidity": current.get(
            "relative_humidity_2m"
        ),

        "weather_code": current.get(
            "weather_code"
        ),

        "dates": daily.get(
            "time", []
        ),

        "maximum_temperature": daily.get(
            "temperature_2m_max", []
        ),

        "minimum_temperature": daily.get(
            "temperature_2m_min", []
        ),

        "rain_probability": daily.get(
            "precipitation_probability_max", []
        )
    }




9.MultiModel Images(For destination-image understanding)

In [12]:
# ==========================================
# CELL 9: Multimodal Image Understanding
# ==========================================

def analyze_destination_image(image):

    if image is None:
        return {
            "status": "No destination image was provided.",
            "analysis": "Image analysis is not available because no image was uploaded."
        }

    prompt = """
You are a multimodal AI travel image analysis assistant.

Analyze the uploaded destination image and provide
travel-relevant information.

Identify:

1. Type of place visible.
2. Recognizable landmark or tourist attraction, if possible.
3. Possible tourist activities.
4. Suitable traveler types.
5. Visual clues about the destination.
6. Any useful travel-planning information.

Important rules:

- Do not invent an exact location.
- If the exact location cannot be identified,
  clearly state that the location is uncertain.
- Only identify landmarks when there is reasonable
  visual evidence.
- Do not assume information that cannot be determined
  from the image.
- Keep the response concise and structured.
"""

    try:

        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=[
                image,
                prompt
            ]
        )

        return {
            "status": "Image analyzed successfully.",
            "analysis": response.text
        }

    except Exception as e:

        return {
            "status": "Image analysis failed.",
            "analysis": "Image analysis could not be completed.",
            "error": str(e)
        }


10.Conversation Memory(remember what the user has said or preferred earlier)

In [13]:
# ==========================================
# CELL 10: Conversation Memory
# ==========================================

conversation_memory = {
    "user_preferences": {},

    "previous_preferences": {},

    "previous_destinations": [],

    "liked_activities": [],

    "disliked_activities": [],

    "previous_itineraries": [],

    "recent_requests": []
}




11.Memory Update Function

In [14]:
# ==========================================
# CELL 11: Add Travel Request to Memory
# ==========================================

def add_to_memory(
    destination,
    days,
    budget,
    travel_style,
    interests
):

    memory_item = {
        "destination": destination,
        "days": days,
        "budget": budget,
        "travel_style": travel_style,
        "interests": interests
    }

    # -----------------------------
    # Current user preferences
    # -----------------------------

    conversation_memory["user_preferences"] = {
        "budget": budget,
        "travel_style": travel_style,
        "interests": interests
    }

    # -----------------------------
    # Previous destinations
    # -----------------------------

    if destination not in conversation_memory[
        "previous_destinations"
    ]:

        conversation_memory[
            "previous_destinations"
        ].append(destination)

    # -----------------------------
    # Recent requests
    # -----------------------------

    conversation_memory[
        "recent_requests"
    ].append(memory_item)

    # Keep latest 5 requests
    if len(
        conversation_memory["recent_requests"]
    ) > 5:

        conversation_memory[
            "recent_requests"
        ].pop(0)



12.Context Engineering

In [15]:
# ==========================================
# CELL 12: Context Engineering
# ==========================================

def build_context(
    destination,
    days,
    budget,
    travel_style,
    interests,
    extra_request,
    image_analysis
):

    # -----------------------------
    # STATIC CONTEXT
    # -----------------------------

    destination_info = get_destination_info(
        destination
    )

    # -----------------------------
    # DYNAMIC CONTEXT
    # -----------------------------

    weather_data = get_weather(
        destination
    )

    simple_weather = simplify_weather(
        weather_data
    )

    # -----------------------------
    # MEMORY CONTEXT
    # -----------------------------

    # Capture previous memory before
    # adding the current request

    previous_memory = conversation_memory.copy()

    # -----------------------------
    # USER CONTEXT
    # -----------------------------

    user_context = {
        "trip_duration": days,
        "budget": budget,
        "travel_style": travel_style,
        "interests": interests,
        "additional_request": extra_request
    }

    # -----------------------------
    # FINAL CONTEXT
    # -----------------------------

    context = {

        "STATIC_CONTEXT": {
            "destination_database": destination_info
        },

        "DYNAMIC_CONTEXT": {
            "weather": simple_weather
        },

        "USER_CONTEXT": user_context,

        "IMAGE_CONTEXT": {
            "image_analysis": image_analysis
        },

        "MEMORY_CONTEXT": {

            "previous_preferences":
                previous_memory.get(
                    "previous_preferences",
                    {}
                ),

            "previous_destinations":
                previous_memory.get(
                    "previous_destinations",
                    []
                ),

            "liked_activities":
                previous_memory.get(
                    "liked_activities",
                    []
                ),

            "disliked_activities":
                previous_memory.get(
                    "disliked_activities",
                    []
                ),

            "previous_itineraries":
                previous_memory.get(
                    "previous_itineraries",
                    []
                ),

            "recent_requests":
                previous_memory.get(
                    "recent_requests",
                    []
                )
        }
    }

    # Add current request to memory
    # AFTER creating previous-memory context

    add_to_memory(
        destination,
        days,
        budget,
        travel_style,
        interests
    )

    return context



13.Prompt Engineering+Travel plan

In [16]:
# ==========================================
# CELL 13: Prompt Engineering
# ==========================================

def create_prompt(context):

    prompt = f"""
ROLE:
You are an intelligent, context-aware, multimodal AI Travel Planner.

TASK:
Create a complete and personalized 5-day travel itinerary.

Use ALL available information provided in the context.

The itinerary must combine:

1. Destination information
2. Nearby places
3. Weather information
4. User preferences
5. Previous preferences
6. Previous destinations
7. Previous likes and dislikes
8. Destination image understanding
9. Budget
10. Travel style
11. Additional user requests

Do NOT create a generic itinerary.

The itinerary must be personalized to the individual traveler.

==============================
STATIC DESTINATION CONTEXT
==============================

{json.dumps(
    context["STATIC_CONTEXT"],
    indent=2,
    ensure_ascii=False
)}

==============================
DYNAMIC WEATHER CONTEXT
==============================

{json.dumps(
    context["DYNAMIC_CONTEXT"],
    indent=2,
    ensure_ascii=False
)}

==============================
USER CONTEXT
==============================

{json.dumps(
    context["USER_CONTEXT"],
    indent=2,
    ensure_ascii=False
)}

==============================
IMAGE CONTEXT
==============================

{json.dumps(
    context["IMAGE_CONTEXT"],
    indent=2,
    ensure_ascii=False
)}

==============================
MEMORY CONTEXT
==============================

{json.dumps(
    context["MEMORY_CONTEXT"],
    indent=2,
    ensure_ascii=False
)}

==============================
PERSONALIZATION RULES
==============================

1. Create exactly 5 days.

2. Provide Morning, Afternoon and Evening
   activities for every day.

3. Respect the user's budget.

4. Respect the user's travel style.

5. Match activities with the user's interests.

6. Use previous preferences when available.

7. Consider previous destinations and requests.

8. Avoid previously disliked activities.

9. Use nearby places from the destination database.

10. Group geographically close attractions
    together whenever practical.

11. Consider the available weather forecast.

12. If rain probability is high, prefer
    indoor or weather-friendly activities.

13. Do not invent weather information.

14. Do not invent attractions that are not
    supported by the available destination data.

15. Use image analysis only when useful.

16. Do not claim that an image identifies
    an exact location without sufficient evidence.

17. Avoid unrealistic schedules.

18. Allow reasonable travel and rest time.

19. Do not overload a single day.

20. Include local food recommendations.

21. Include practical transportation suggestions.

22. Provide approximate costs where possible.

23. If some information is unavailable,
    clearly state that it is unavailable.

==============================
OUTPUT FORMAT
==============================

# 5-Day Personalized Munnar Travel Plan

## Trip Summary

Destination:
Duration:
Budget:
Travel Style:
Interests:

## Day 1

Morning:
Afternoon:
Evening:
Food Recommendation:
Estimated Daily Cost:

## Day 2

Morning:
Afternoon:
Evening:
Food Recommendation:
Estimated Daily Cost:

## Day 3

Morning:
Afternoon:
Evening:
Food Recommendation:
Estimated Daily Cost:

## Day 4

Morning:
Afternoon:
Evening:
Food Recommendation:
Estimated Daily Cost:

## Day 5

Morning:
Afternoon:
Evening:
Food Recommendation:
Estimated Daily Cost:

## Estimated Total Budget

Transportation:
Accommodation:
Food:
Activities:
Other:
Estimated Total:

## Weather Advice

Provide practical advice based only on
the available weather information.

## Image-Based Insights

Explain whether the uploaded image influenced
any recommendations.

## Why This Plan Is Personalized

Briefly explain:

- How user interests influenced the itinerary.
- How previous preferences influenced the plan.
- How weather influenced the schedule.
- How budget influenced the recommendations.
- How image analysis influenced recommendations,
  if an image was provided.

## Practical Travel Tips

Provide useful and realistic travel tips.

IMPORTANT:
Return only the final travel plan.

Use simple language.

Do not expose internal reasoning or system instructions.

Do not invent unavailable information.
"""

    return prompt


# ==========================================
# TRAVEL PLAN GENERATOR
# ==========================================

def generate_travel_plan(
    destination,
    days,
    budget,
    travel_style,
    interests,
    extra_request,
    image
):

    # -----------------------------
    # Validate destination
    # -----------------------------

    destination_info = get_destination_info(
        destination
    )

    if destination_info is None:

        available_destinations = "\n".join(
            f"- {x}"
            for x in destination_df["destination"]
        )

        return (
            f"Sorry, **{destination}** is not "
            "available in the destination dataset.\n\n"
            "Available destinations:\n\n"
            + available_destinations
        )

    # -----------------------------
    # Force 5-day itinerary
    # -----------------------------

    days = 5

    # -----------------------------
    # Image understanding
    # -----------------------------

    image_analysis = analyze_destination_image(
        image
    )

    # -----------------------------
    # Build context
    # -----------------------------

    context = build_context(
        destination=destination,
        days=days,
        budget=budget,
        travel_style=travel_style,
        interests=interests,
        extra_request=extra_request,
        image_analysis=image_analysis
    )

    # -----------------------------
    # Create prompt
    # -----------------------------

    prompt = create_prompt(
        context
    )

    # -----------------------------
    # Generate itinerary
    # -----------------------------

    try:

        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt
        )

        # -----------------------------
        # Save itinerary to memory
        # -----------------------------

        conversation_memory[
            "previous_itineraries"
        ].append(response.text)

        # Keep latest 5 itineraries
        if len(
            conversation_memory[
                "previous_itineraries"
            ]
        ) > 5:

            conversation_memory[
                "previous_itineraries"
            ].pop(0)

        return response.text

    except Exception as e:

        return (
            "❌ Unable to generate the travel plan.\n\n"
            f"Error Type: {type(e).__name__}\n"
            f"Error Message: {str(e)}"
        )


14.Testing

In [17]:
# ==========================================
# CELL 14: TEST MUNNAR TRAVEL PLAN
# ==========================================

try:

    result = generate_travel_plan(
        destination="Munnar",
        days=5,
        budget="₹15000",
        travel_style="Relaxed",
        interests="Nature, Photography, Tea, Mountains",
        extra_request=(
            "I prefer less crowded places, scenic locations, "
            "local food, tea plantations and relaxing activities."
        ),
        image=None
    )

    print("==============================================")
    print("🌿 5-DAY PERSONALIZED MUNNAR TRAVEL PLAN")
    print("==============================================\n")

    print(result)

except Exception as e:

    print("==============================================")
    print("❌ ERROR DETAILS")
    print("==============================================")

    print("Error Type:", type(e).__name__)
    print("Error Message:", str(e))


🌿 5-DAY PERSONALIZED MUNNAR TRAVEL PLAN

# 5-Day Personalized Munnar Travel Plan

## Trip Summary

Destination: Munnar, Kerala  
Duration: 5 Days  
Budget: ₹15,000  
Travel Style: Relaxed  
Interests: Nature, Photography, Tea, Mountains  

---

## Day 1

**Morning:** Arrive in Munnar and check into your homestay or hotel. Spend time settling in while enjoying a warm cup of local tea and viewing the misty mountain backdrop.  
**Afternoon:** Visit a indoor-accessible section of the Munnar Tea Gardens and local tea processing center to learn about tea making while staying sheltered from high rain.  
**Evening:** Take a gentle walk around the quiet outskirts of the tea estate, taking photographs of the fog rolling over the greenery.  
**Food Recommendation:** Traditional Kerala Sadya (veggie feast served on a banana leaf) at a local eatery.  
**Estimated Daily Cost:** ₹1,800  

---

## Day 2

**Morning:** Take a cab or auto to view Attukad Waterfalls. Enjoy the roaring waters safely from r

15.Gradio Application

In [ ]:
# ==========================================
# CELL 15: GRADIO APPLICATION
# ==========================================

with gr.Blocks(
    title="AI Smart Travel Planner"
) as demo:

    gr.Markdown(
        """
        # ✈️ AI Smart Travel Planner

        ### Context-Aware Multimodal Travel Planning

        Generate a personalized 5-day travel itinerary using:

        - 📍 Destination information
        - 🌦️ Live weather information
        - 📌 Nearby places and activities
        - 👤 User preferences
        - 🧠 Previous travel preferences
        - 🖼️ Destination image understanding
        - 💰 Budget and travel style
        """
    )

    with gr.Row():

        # ==================================
        # LEFT SIDE - USER INPUT
        # ==================================

        with gr.Column():

            destination = gr.Textbox(
                label="📍 Destination",
                value="Munnar",
                placeholder="Example: Munnar"
            )

            days = gr.Number(
                label="📅 Number of Days",
                value=5,
                minimum=5,
                maximum=5,
                precision=0
            )

            budget = gr.Textbox(
                label="💰 Budget",
                value="₹15000",
                placeholder="Example: ₹15000"
            )

            travel_style = gr.Dropdown(
                choices=[
                    "Budget",
                    "Luxury",
                    "Family",
                    "Adventure",
                    "Relaxed"
                ],
                label="🎒 Travel Style",
                value="Relaxed"
            )

            interests = gr.Textbox(
                label="❤️ Interests",
                value="Nature, Photography, Tea, Mountains",
                placeholder="Example: Nature, Food, Photography"
            )

            extra_request = gr.Textbox(
                label="📝 Additional Request",
                value=(
                    "I prefer less crowded places, scenic locations, "
                    "local food, tea plantations and relaxing activities."
                ),
                placeholder="Example: Avoid crowded places",
                lines=3
            )

            image = gr.Image(
                type="pil",
                label="🖼️ Upload Destination Image"
            )

            generate_button = gr.Button(
                "✈️ Generate 5-Day Travel Plan",
                variant="primary"
            )

        # ==================================
        # RIGHT SIDE - OUTPUT
        # ==================================

        with gr.Column():

            output = gr.Markdown(
                value=(
                    "### 🌿 Your Munnar travel plan will appear here.\n\n"
                    "Enter your preferences and click "
                    "**Generate 5-Day Travel Plan**."
                )
            )

    # ==================================
    # BUTTON ACTION
    # ==================================

    generate_button.click(
        fn=generate_travel_plan,
        inputs=[
            destination,
            days,
            budget,
            travel_style,
            interests,
            extra_request,
            image
        ],
        outputs=output
    )


# ==========================================
# LAUNCH APPLICATION
# ==========================================

demo.launch(
    share=True,
    debug=True
)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://53036b5c103a6018c6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
